In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [2]:
# starting with self attention expl
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)

torch.Size([4, 8, 2])


In [3]:
#bag of words for x
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] #xprev.shape == t,C i.e. 1,2 .. 8,2
        xbow[b,t] = torch.mean(xprev, 0)
print(xbow.shape)
# first entries are equal, then every next xbow is an average of itself and prev entries
print("x:", x[0], "\nbow:",  xbow[0]) 


torch.Size([4, 8, 2])
x: tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]]) 
bow: tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [4]:
# there is a way to accum the same value in xbow using matrix multiplication
# it requires triangular matrix:
torch.manual_seed(42)
trl = torch.tril(torch.ones(3,3))
print(trl)
tens = torch.randint(0, 10, (3,2)).float()
print(tens)
print(trl @ tens)
print("--")
# example with triangular matrix of ones produces a sum of current elem and prev
# to get mean of it, we need to modify triangular matrix:
trl = trl / torch.sum(trl, 1, keepdim=True)
print(trl)
print(trl @ tens)


tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])
--
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [5]:
# so to calc xbow:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
#original dims here: T,T @ B,T,C. but since they do not match, first entry will be stretched
# with an extra dimension B, so it becomes: [B],T,T @ [B],T,C -> B,T,C
xbow2 = wei @ x 
print("bows match:", torch.allclose(xbow2, xbow))

bows match: True


In [6]:
# there is another way to calc bows:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei)
wei = F.softmax(wei, dim=-1)
print(wei)

xbow3 = wei @ x
print("bows match:", torch.allclose(xbow3, xbow))

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
bows match: Tr

In [7]:
# self attention implementation example
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
print(query)
k = key(x)     # (B,T,16)
print(k.shape)
q = query(x)   # (B,T,16)
wei = q @ k.transpose(-2, -1) #(B,T,16) @ (B,16,T) -> (B,T,T)

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
#out = wei @ x
v = value(x)
out = wei @ v
print(out.shape)


Linear(in_features=32, out_features=16, bias=False)
torch.Size([4, 8, 16])
torch.Size([4, 8, 16])
